# NosoGraph — end-to-end pipeline replication (demo)

Step-by-step reproduction of the `assembly-qc-iden` integration test: from raw Nanopore
long reads to **Neo4j-ready knowledge-graph CSVs**, in tutorial order — **(1) Kraken2
metagenomic classification, then (2) bacterial assembly + QC + identification**.

Pipelines exercised: `kraken2-classify`, `bacterial-assembly` (Flye `nanopore-hq`),
`assembly-qc-iden` (QUAST + CheckM2 + BLAST), and the NosoGraph KG exporters.

> Shell cells use the `bash` magic. Nextflow manages every tool through micromamba conda
> envs, so there is **nothing to install by hand** beyond Nextflow + micromamba.

## 0. Configuration

Edit these paths for your host, then run every cell top-to-bottom.

In [ ]:
import os
os.environ['REPO']        = '/path/to/NosoGraph'
os.environ['FASTQ_DIR']   = '/path/to/fastq'
os.environ['KRAKEN2_DB']  = '/mnt/central/KRAKEN2/kraken2_standard/08G_table'
os.environ['BLAST_DB']    = '/mnt/central/BLAST/Silva_SSU/SILVA_138.2_SSURef'
os.environ['CHECKM2_DB']  = '/mnt/central/CHECKM2DB/CheckM2_database/uniref100.KO.1.dmnd'
os.environ['OUTROOT']     = './results'
# Reuse one conda-env cache across all runs (envs are built once, on first use):
os.environ['NXF_CONDA_CACHEDIR'] = './nf-conda-cache'
print('config set')

## 1. Environment check

Confirm Nextflow, Java and micromamba are present. (No conda/mamba required.)

In [ ]:
%%bash
cd "$REPO"
nextflow -version | sed -n '2,4p'
java -version 2>&1 | head -1
micromamba --version | sed 's/^/micromamba /'

## 2. Inputs & databases

Sanity-check that the reads and the three reference databases are reachable.

In [ ]:
%%bash
ls -lh "$FASTQ_DIR"/CI16.long.fq.gz "$FASTQ_DIR"/CI19.long.fq.gz
echo '--- kraken2 db (needs hash.k2d/opts.k2d/taxo.k2d) ---'; ls "$KRAKEN2_DB"/{hash,opts,taxo}.k2d
echo '--- blast db ---'; ls "${BLAST_DB}".n* | head -3
echo '--- checkm2 db ---'; ls -lh "$CHECKM2_DB"

## 3. Validate pipeline wiring (no data, no conda)

`-stub-run -profile test` compiles the whole DAG and checks every process connection
without real inputs or tools. Expect `[SUCCESS]` for both pipelines.

In [ ]:
%%bash
cd "$REPO"
nextflow run main.nf -stub-run -profile test --pipeline bacterial-assembly \
  --sample_id CI16 --assembler flye --tech nanopore-hq --pilon_iter 0 \
  --long_reads dummy.fq.gz --blast_db dummy --checkm2_db dummy.dmnd \
  --outdir /tmp/nf_stub_ba 2>&1 | tail -4
echo '======'
nextflow run main.nf -stub-run -profile test --pipeline metagenomics \
  --sample_id CI16 --long_reads dummy.fq.gz --kraken2_db dummy --outdir /tmp/nf_stub_meta 2>&1 | tail -4

## 4. Step 1 — metagenomic classification (Kraken2)

Classify the long reads against the **8 GB Standard** Kraken2 DB and export the
pathogen-ID subgraph CSVs. We override `--kraken2_mem` to `12 GB` (the 8 GB hash fits;
the module's 64 GB default would be refused by the local executor on a 39 GB host).

> First run loads the 8 GB hash from disk (slow on NFS); the second sample is fast because
> the hash is warm in the OS page cache.

In [ ]:
%%bash
cd "$REPO"
for sid in CI16 CI19; do
  echo "=== METAGENOMICS $sid ==="
  nextflow run main.nf -profile micromamba -ansi-log false -resume \
    -work-dir "./work/$sid" \
    --pipeline metagenomics --sample_id "$sid" \
    --long_reads "$FASTQ_DIR/$sid.long.fq.gz" \
    --kraken2_db "$KRAKEN2_DB" --kraken2_mem '12 GB' --threads 16 \
    --outdir "$OUTROOT/$sid" 2>&1 | tail -4
done

### Inspect classification result

In [ ]:
%%bash
for sid in CI16 CI19; do
  echo "=== $sid: top species (Kraken2 rank S) ==="
  awk -F'\t' '$4=="S"' "$OUTROOT/$sid/kraken2/$sid.kraken2.report.txt" | sort -t$'\t' -k1 -nr | head -3
  echo "--- taxonomic_classification.csv ---"; cat "$OUTROOT/$sid/$sid/kg/taxonomic_classification.csv"
done

## 5. Step 2 — assembly + QC + identification + KG export

Flye (`--nano-hq`) → Racon polish → QUAST / CheckM2 / BLAST → KG CSV export. Only long
reads are available, so Pilon is disabled (`--pilon_iter 0`); completeness/contamination
and per-contig accessions come from `assembly-qc-iden`. The metagenomics and assembly
CSVs land in the **same** `kg/` bundle per sample.

In [ ]:
%%bash
cd "$REPO"
for sid in CI16 CI19; do
  echo "=== ASSEMBLY $sid ==="
  nextflow run main.nf -profile micromamba -ansi-log false -resume \
    -work-dir "./work/$sid" \
    --pipeline bacterial-assembly --sample_id "$sid" \
    --long_reads "$FASTQ_DIR/$sid.long.fq.gz" \
    --assembler flye --tech nanopore-hq --pilon_iter 0 --racon_iter 1 \
    --blast_db "$BLAST_DB" --checkm2_db "$CHECKM2_DB" --threads 16 \
    --outdir "$OUTROOT/$sid" 2>&1 | tail -4
done

### Inspect assembly QC + identification

In [ ]:
%%bash
for sid in CI16 CI19; do
  echo "=== $sid assembly.csv (CheckM2 completeness/contamination) ==="; cat "$OUTROOT/$sid/$sid/kg/assembly.csv"
  echo "--- contigs.csv (name, BLAST accession, length, coverage, circular) ---"
  cut -d, -f3,4,5,6,7 "$OUTROOT/$sid/$sid/kg/contigs.csv"
done

## 6. The knowledge-graph CSV bundle

Seven CSVs per sample, ready for the `LOAD CSV` templates in
`assets/nosograph_cypher_templates.csv`.

In [ ]:
%%bash
for sid in CI16 CI19; do
  echo "=== $OUTROOT/$sid/$sid/kg ==="; ls -1 "$OUTROOT/$sid/$sid/kg"/*.csv
done

## 7. Load into Neo4j (optional)

Import the bundle with the saved-query templates in
[`assets/nosograph_cypher_templates.csv`](../../assets/nosograph_cypher_templates.csv):
copy each sample's `kg/*.csv` into the Neo4j `import/` directory, then run the
`LOAD DATA` steps (assembly steps for `sample/assembly/contigs/biodata_files`, and steps
17–19 for the metagenomics `meta_reads/taxonomic_classification/taxa`). Finish with the
**QUERIES → Pathogens detected per sample** template to confirm the graph.